In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from config import DATA_DIR, RESULTS_DIR, FIGURS_DIR, METADATA_DIR
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    roc_auc_score,
    f1_score,
    accuracy_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import train_test_split
from model import TabularClassifier
from adni_data import ADNITabularDataset
from torch.utils.data import DataLoader
from config import NUMERIC_PREDICTORS, CATEGORICAL_PREDICTORS,EXPERIMENTS
from util import preprocess_adni, compute_class_weights_effective_num,create_or_load_splits
import matplotlib.pyplot as plt 

In [22]:
"""
Build the LaTeX results table dynamically from the ADNI test-result dataframes.

Assumes the following are already loaded in your session (as in your snippet):

tdf_all, tdf_CN_AD, tdf_CN_MCI, tdf_MCI_AD
vdf_all, vdf_CN_AD, vdf_CN_MCI, vdf_MCI_AD
m_atten_df_all, m_atten_CN_AD, m_atten_CN_MCI, m_atten_MCI_AD
m_concat_df_all, m_concat_CN_AD, m_concat_CN_MCI, m_concat_MCI_AD

Each dataframe has columns:
['Model', 'Metric', 'Point_Est', 'CI_Lower', 'CI_Upper', 'CI_Level', 'N_Bootstrap']
"""



tdf_all = pd.read_csv("results/tabular_test_results.csv")
tdf_CN_AD = pd.read_csv("results/tabular_test_results__CN_AD.csv")
tdf_CN_MCI = pd.read_csv("results/tabular_test_results__CN_MCI.csv")
tdf_MCI_AD = pd.read_csv("results/tabular_test_results__MCI_AD.csv")

vdf_all = pd.read_csv("results/vision_test_results.csv")
vdf_CN_AD = pd.read_csv("results/vision_test_results__CN_AD.csv")
vdf_CN_MCI = pd.read_csv("results/vision_test_results__CN_MCI.csv")
vdf_MCI_AD = pd.read_csv("results/vision_test_results__MCI_AD.csv")


m_atten_df_all = pd.read_csv("results/multimodal_test_results_cross_attn.csv")
m_atten_CN_AD = pd.read_csv("results/multimodal_test_results_cross_attn__CN_AD.csv")
m_atten_CN_MCI = pd.read_csv("results/multimodal_test_results_cross_attn__CN_MCI.csv")
m_atten_MCI_AD = pd.read_csv("results/multimodal_test_results_cross_attn__MCI_AD.csv")


m_concat_df_all = pd.read_csv("results/multimodal_test_results_concat.csv")
m_concat_CN_AD = pd.read_csv("results/multimodal_test_results_concat__CN_AD.csv")
m_concat_CN_MCI = pd.read_csv("results/multimodal_test_results_concat__CN_MCI.csv")
m_concat_MCI_AD = pd.read_csv("results/multimodal_test_results_concat__MCI_AD.csv")



# ---------------------------------------------------------------------------
# 1. Wire up which dataframes belong to which (task, modality)
# ---------------------------------------------------------------------------
TASKS = [
    ("CN vs MCI vs AD", {
        "Tabular-only":                 tdf_all,
        "Vision-only":                  vdf_all,
        "Multimodal (Concat)":          m_concat_df_all,
        "Multimodal (Cross-Attention)": m_atten_df_all,
    }),
    ("CN vs AD", {
        "Tabular-only":                 tdf_CN_AD,
        "Vision-only":                  vdf_CN_AD,
        "Multimodal (Concat)":          m_concat_CN_AD,
        "Multimodal (Cross-Attention)": m_atten_CN_AD,
    }),
    ("CN vs MCI", {
        "Tabular-only":                 tdf_CN_MCI,
        "Vision-only":                  vdf_CN_MCI,
        "Multimodal (Concat)":          m_concat_CN_MCI,
        "Multimodal (Cross-Attention)": m_atten_CN_MCI,
    }),
    ("MCI vs AD", {
        "Tabular-only":                 tdf_MCI_AD,
        "Vision-only":                  vdf_MCI_AD,
        "Multimodal (Concat)":          m_concat_MCI_AD,
        "Multimodal (Cross-Attention)": m_atten_MCI_AD,
    }),
]

MODEL_ORDER = ["Tabular-only", "Vision-only", "Multimodal (Concat)", "Multimodal (Cross-Attention)"]

# Exact metric names as they appear in the 'Metric' column (case-insensitive match).
AUC_METRIC = "AUROC"
F1_METRIC = "Micro F1"


# ---------------------------------------------------------------------------
# 2. Helpers
# ---------------------------------------------------------------------------
def find_metric_row(df: pd.DataFrame, metric_name: str) -> pd.Series:
    """Return the single row whose Metric exactly matches `metric_name` (case-insensitive)."""
    matches = df[df["Metric"].str.lower() == metric_name.lower()]
    if matches.empty:
        raise ValueError(
            f"No metric named '{metric_name}' found. "
            f"Available metrics: {sorted(df['Metric'].unique())}"
        )
    if len(matches) > 1:
        raise ValueError(
            f"Multiple rows matched '{metric_name}' — this df likely has more than one 'Model'.\n"
            f"{matches[['Model', 'Metric']]}"
        )
    return matches.iloc[0]


def format_ci(row: pd.Series, decimals: int = 3) -> str:
    """Format a row as 'point (lower--upper)'."""
    return (
        f"{row['Point_Est']:.{decimals}f} "
        f"({row['CI_Lower']:.{decimals}f}--{row['CI_Upper']:.{decimals}f})"
    )


def build_metric_strings(df: pd.DataFrame) -> tuple[str, str]:
    auc_row = find_metric_row(df, AUC_METRIC)
    f1_row = find_metric_row(df, F1_METRIC)
    return format_ci(auc_row), format_ci(f1_row)


# ---------------------------------------------------------------------------
# 3. Build the LaTeX
# ---------------------------------------------------------------------------
def build_latex_table(tasks=TASKS, model_order=MODEL_ORDER) -> str:
    lines = [
        r"\begin{table*}[!ht]",
        r"\centering",
        r"\caption{",
        r"Test-set performance across ADNI classification tasks.",
        r"Values represent AUC--ROC and Micro F1 scores with 95\% bootstrap confidence intervals.",
        r"}",
        r"\label{tab:adni_multimodal_results}",
        r"\small",
        r"\setlength{\tabcolsep}{8pt}",
        r"\begin{tabular}{llcc}",
        r"\toprule",
        r"\textbf{Task} &",
        r"\textbf{Model} &",
        r"\textbf{AUC--ROC (95\% CI)} &",
        r"\textbf{Micro F1 (95\% CI)} \\",
        r"\midrule",
    ]

    for i, (task_label, model_dfs) in enumerate(tasks):
        lines.append(f"\\multirow{{{len(model_order)}}}{{*}}{{{task_label}}}")
        for model_label in model_order:
            df = model_dfs[model_label]
            auc_str, f1_str = build_metric_strings(df)
            lines.append(f"& {model_label}")
            lines.append(f"& {auc_str}")
            lines.append(f"& {f1_str} \\\\")
        if i < len(tasks) - 1:
            lines.append(r"\midrule")

    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table*}",
    ]

    return "\n".join(lines)



latex_table = build_latex_table()
print(latex_table)

# # Optionally write straight to a .tex file:
# with open("adni_multimodal_results_table.tex", "w") as f:
#     f.write(latex_table)

\begin{table*}[!ht]
\centering
\caption{
Test-set performance across ADNI classification tasks.
Values represent AUC--ROC and Micro F1 scores with 95\% bootstrap confidence intervals.
}
\label{tab:adni_multimodal_results}
\small
\setlength{\tabcolsep}{8pt}
\begin{tabular}{llcc}
\toprule
\textbf{Task} &
\textbf{Model} &
\textbf{AUC--ROC (95\% CI)} &
\textbf{Micro F1 (95\% CI)} \\
\midrule
\multirow{4}{*}{CN vs MCI vs AD}
& Tabular-only
& 0.894 (0.875--0.912)
& 0.682 (0.645--0.721) \\
& Vision-only
& 0.764 (0.736--0.791)
& 0.567 (0.524--0.607) \\
& Multimodal (Concat)
& 0.956 (0.941--0.968)
& 0.830 (0.791--0.864) \\
& Multimodal (Cross-Attention)
& 0.926 (0.907--0.942)
& 0.739 (0.695--0.781) \\
\midrule
\multirow{4}{*}{CN vs AD}
& Tabular-only
& 0.997 (0.992--0.999)
& 0.972 (0.954--0.988) \\
& Vision-only
& 0.939 (0.912--0.961)
& 0.837 (0.800--0.877) \\
& Multimodal (Concat)
& 1.000 (1.000--1.000)
& 0.955 (0.925--0.985) \\
& Multimodal (Cross-Attention)
& 1.000 (1.000--1.000)
& 0.985 (0.

In [3]:
df_processed = pd.read_csv(Path(RESULTS_DIR, "ADNI_Combined_Multimodal_FIXED.csv"))

df_processed['Group'].value_counts()

Group
MCI    3026
CN     2240
AD     1213
Name: count, dtype: int64

In [8]:
df_processed.columns

Index(['subject_id', 'processed_path', 'RID', 'COLPROT', 'ORIGPROT', 'PTID',
       'SITE', 'VISCODE', 'EXAMDATE', 'DX_bl',
       ...
       'FDG_bl', 'PIB_bl', 'AV45_bl', 'FBB_bl', 'Years_bl', 'Month_bl',
       'Month', 'M', 'update_stamp', 'Group'],
      dtype='object', length=119)

In [10]:
df_processed['Sex']

0         Male
1         Male
2       Female
3       Female
4       Female
         ...  
6474      Male
6475    Female
6476    Female
6477    Female
6478    Female
Name: Sex, Length: 6479, dtype: object

In [ ]:
# df_processed = pd.read_csv(Path(RESULTS_DIR, "ADNI_Combined_Multimodal_FIXED.csv"))



# # Define target column and new base path
# NEW_BASE = Path("/home/ybrima/Data/ADNI_Data/ADNI_PROCESSED")
# col_name = df_processed.columns[1]  # or use the exact column name like 'processed_path'

# # # Rebase paths using pathlib
# # df_processed[col_name] = df_processed[col_name].apply(
# #     lambda p: str(NEW_BASE / Path(p).name)
# # )

# # # Verify first row
# # print(df_processed[col_name].iloc[0])


# col_name = df_processed.columns[1]  # or your exact column name

# # 1. Rebase paths
# df_processed[col_name] = df_processed[col_name].apply(
#     lambda p: str(NEW_BASE / Path(p).name)
# )

# # 2. Keep only rows where the rebased file actually exists on disk
# initial_count = len(df_processed)
# df_processed = df_processed[
#     df_processed[col_name].apply(lambda p: Path(p).exists())
# ].reset_index(drop=True)

# # 3. Print verification stats
# print(
#     f"Retained {len(df_processed)} out of {initial_count} rows with existing files."
# )
# if not df_processed.empty:
#     print("Sample rebased path:", df_processed[col_name].iloc[0])


# output_path = Path(RESULTS_DIR, "ADNI_Combined_Multimodal_FINAL.csv")
# df_processed.to_csv(output_path, index=False)


In [ ]:
# !python tab_trainer.py 

In [18]:
# ==============================================================================
# CONFIGURATION
# ==============================================================================
# None -> use all classes
# ["CN", "AD"] -> binary experiment
# ["CN", "MCI"] -> binary experiment
# ["MCI", "AD"] -> binary experiment

experiment = 0
selected_classes = EXPERIMENTS[experiment]

print(f"Running experiment {experiment}")
print(f"Selected classes: {selected_classes or 'All classes'}")

# Three-class experiment (default)
train_df, val_df, test_df, suffix = create_or_load_splits(
    df_processed,
    RESULTS_DIR,
    selected_classes=selected_classes,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


data    = preprocess_adni(train_df, val_df, test_df, keep_paths=True)
CLASSES = data["label_encoder"].classes_

test_dataset  = ADNITabularDataset(test_df,  data["X_test"],  CLASSES)

batch_size = 32

test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False,
    num_workers=4, pin_memory=True,
)

# Sanity check
X_sample, y_sample = next(iter(test_loader))
print(f"\nSample batch — X: {X_sample.shape} | y: {y_sample.shape}")
print(f"Unique labels in batch: {y_sample.unique().tolist()}")


Running experiment 0
Selected classes: All classes
Loading existing splits...
Using device: cuda
Train: 5174 | Val: 714 | Test: 591
Splits — Train: 5174 | Val: 714 | Test: 591
Dropping 1 column(s) exceeding 60% missingness: ['MOCA_bl']


/home/ybrima/miniconda3/envs/mxai/lib/python3.12/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Classes: ['AD' 'CN' 'MCI']
Label distribution — Train: [1026 1753 2395] | Val: [ 91 258 365] | Test: [ 96 229 266]
Feature matrix — Train: (5174, 13) | Val: (714, 13) | Test: (591, 13)

Sample batch — X: torch.Size([32, 13]) | y: torch.Size([32])
Unique labels in batch: [0, 1, 2]


In [ ]:
if suffix != "":
    suffix = f"_{suffix}"

model = TabularClassifier(
    n_features=X_sample.shape[1],
    num_classes=len(CLASSES),
    dropout=0.6,
).to(DEVICE)
print(f"\n{model}\n")

model.eval()

# Load best checkpoint before evaluating
model.load_state_dict(torch.load(Path("./savedmodels", f"tabular_classifier_final{suffix}.pth"), map_location=DEVICE))


In [ ]:
# !pip install -q shap captum

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import f1_score

# Publication-quality defaults
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 1.0,
    "savefig.bbox": "tight",
})
sns.set_palette("colorblind")

FIG_DIR = Path(RESULTS_DIR, "figures")
FIG_DIR.mkdir(exist_ok=True, parents=True)

def save_fig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight", dpi=300)
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight", dpi=300)
    print(f"Saved {name}.pdf / .png")

In [ ]:
FEATURE_NAMES = list(data["feature_names"])
n_features = X_sample.shape[1]
assert len(FEATURE_NAMES) == n_features, (
    f"Mismatch: {len(FEATURE_NAMES)} names vs {n_features} model input features"
)
print(f"{len(FEATURE_NAMES)} features: {FEATURE_NAMES}")

model.eval()  # ensure dropout is off before any interpretability step

# Ordered (non-shuffled) test tensors, aligned with test_df rows
X_test_t = torch.as_tensor(np.asarray(data["X_test"]), dtype=torch.float32).to(DEVICE)
y_test_t = torch.stack([test_dataset[i][1] for i in range(len(test_dataset))]).to(DEVICE)

# Background set for SHAP — sample from TRAIN, not test
X_train_t = torch.as_tensor(np.asarray(data["X_train"]), dtype=torch.float32).to(DEVICE)
rng = np.random.default_rng(42)
bg_idx = rng.choice(X_train_t.shape[0], size=min(100, X_train_t.shape[0]), replace=False)
background = X_train_t[bg_idx]

print(f"X_test_t: {X_test_t.shape} | y_test_t: {y_test_t.shape} | background: {background.shape}")

In [ ]:
@torch.no_grad()
def predict_labels(model, X):
    logits = model(X)
    return logits.argmax(dim=1).cpu().numpy()

@torch.no_grad()
def macro_f1(model, X, y):
    preds = predict_labels(model, X)
    return f1_score(y.cpu().numpy(), preds, average="macro")

def permutation_importance(model, X, y, feature_names, n_repeats=20, seed=0):
    rng = np.random.default_rng(seed)
    baseline = macro_f1(model, X, y)
    n_samples, n_feat = X.shape
    drops = np.zeros((n_repeats, n_feat))

    for r in range(n_repeats):
        for j in range(n_feat):
            X_perm = X.clone()
            perm_idx = torch.as_tensor(rng.permutation(n_samples))
            X_perm[:, j] = X[perm_idx, j]
            drops[r, j] = baseline - macro_f1(model, X_perm, y)

    return pd.DataFrame({
        "feature": feature_names,
        "importance_mean": drops.mean(axis=0),
        "importance_std": drops.std(axis=0),
    }).sort_values("importance_mean", ascending=False).reset_index(drop=True)

perm_df = permutation_importance(model, X_test_t, y_test_t, FEATURE_NAMES, n_repeats=20)
perm_df.to_csv(Path(RESULTS_DIR, "permutation_importance.csv"), index=False)
perm_df.head(15)

In [ ]:
def plot_permutation_importance(perm_df, top_n=20):
    df = perm_df.head(top_n).iloc[::-1]
    fig, ax = plt.subplots(figsize=(7, 0.35 * len(df) + 1.5))
    ax.barh(df["feature"], df["importance_mean"], xerr=df["importance_std"],
            color=sns.color_palette("colorblind")[0], edgecolor="black", linewidth=0.6,
            error_kw={"elinewidth": 1, "capsize": 3})
    ax.set_xlabel("Decrease in macro-F1 (permutation importance)")
    ax.set_title("Global Feature Importance — Permutation")
    fig.tight_layout()
    return fig

fig = plot_permutation_importance(perm_df, top_n=20)
plt.savefig("./figures/permutation_importance.pdf", bbox_inches="tight", dpi=300)
# save_fig(fig, "permutation_importance")
plt.show()

In [ ]:
import shap

model.eval()
explainer = shap.GradientExplainer(model, background)
shap_values = explainer.shap_values(X_test_t)

# Normalize output shape across shap versions
if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    shap_values = [shap_values[:, :, c] for c in range(shap_values.shape[2])]

X_test_np = X_test_t.cpu().numpy()



for c_idx, c_name in enumerate(CLASSES):
    fig = plt.figure(figsize=(7, 0.3 * len(FEATURE_NAMES) + 2))
    shap.summary_plot(
        shap_values[c_idx], X_test_np, feature_names=FEATURE_NAMES,
        show=False, plot_size=None
    )
    plt.title(f"SHAP Summary — Class: {c_name}")
    plt.tight_layout()
    # save_fig(plt.gcf(), f"shap_summary_{c_name}")
    plt.savefig(f"./figures/shap_summary_{c_name}.pdf", bbox_inches="tight", dpi=300)

    plt.show()

In [ ]:
importance_matrix = np.stack(
    [np.abs(shap_values[c]).mean(axis=0) for c in range(len(CLASSES))],
    axis=0
)

imp_df = pd.DataFrame(importance_matrix, index=CLASSES, columns=FEATURE_NAMES)
imp_df.to_csv(Path(RESULTS_DIR, "shap_importance_by_class.csv"))

top_feats = imp_df.mean(axis=0).sort_values(ascending=False).head(20).index
fig, ax = plt.subplots(figsize=(0.5 * len(top_feats) + 2, 0.5 * len(CLASSES) + 2))
sns.heatmap(imp_df[top_feats], cmap="viridis", ax=ax, cbar_kws={"label": "Mean |SHAP value|"},
            linewidths=0.4, linecolor="white")
ax.set_xlabel("Feature")
ax.set_ylabel("Class")
ax.set_title("Per-Class Feature Importance (SHAP)")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
fig.tight_layout()
# save_fig(fig, "shap_importance_heatmap_by_class")
plt.savefig("./figures/shap_importance_heatmap_by_class.pdf", bbox_inches="tight", dpi=300)

plt.show()

In [ ]:
from captum.attr import IntegratedGradients

ig = IntegratedGradients(model)
baseline_ig = torch.zeros_like(X_test_t[:1])  # zero baseline; swap to X_train_t.mean(0, keepdim=True) if features aren't standardized

ig_matrix = np.zeros((len(CLASSES), len(FEATURE_NAMES)))
for c_idx in range(len(CLASSES)):
    attributions, delta = ig.attribute(
        X_test_t, baselines=baseline_ig.expand_as(X_test_t),
        target=c_idx, n_steps=50, return_convergence_delta=True
    )
    ig_matrix[c_idx] = attributions.abs().mean(dim=0).detach().cpu().numpy()
    print(f"Class {CLASSES[c_idx]}: mean convergence delta = {delta.abs().mean().item():.4f}")

ig_df = pd.DataFrame(ig_matrix, index=CLASSES, columns=FEATURE_NAMES)
ig_df.to_csv(Path(RESULTS_DIR, "integrated_gradients_by_class.csv"))


fig, ax = plt.subplots(figsize=(0.5 * len(top_feats) + 2, 0.5 * len(CLASSES) + 2))
sns.heatmap(ig_df[top_feats], cmap="magma", ax=ax, cbar_kws={"label": "Mean |IG attribution|"},
            linewidths=0.4, linecolor="white")
ax.set_xlabel("Feature")
ax.set_ylabel("Class")
ax.set_title("Per-Class Feature Importance (Integrated Gradients)")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
fig.tight_layout()
plt.savefig("./figures/ig_importance_heatmap_by_class.pdf", bbox_inches="tight", dpi=300)
# save_fig(fig, "ig_importance_heatmap_by_class")
plt.show()

In [ ]:
rank_shap = imp_df.mean(axis=0).rank(ascending=False)
rank_ig = ig_df.mean(axis=0).rank(ascending=False)
corr = rank_shap.corr(rank_ig, method="spearman")
print(f"Spearman rank correlation between SHAP and IG global importance: {corr:.3f}")

# 3D MRI Model Explainability

In [ ]:
from sklearn.model_selection import train_test_split
from config import DATA_DIR, RESULTS_DIR, FIGURS_DIR, METADATA_DIR
import pandas as pd
import numpy as np
from pathlib import Path
import SimpleITK as sitk
from monai.networks.nets import DenseNet121
from torch.utils.data import DataLoader
import torch.optim as optim
from data import ADNIDataset,ADNIDatasetLite
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from sklearn.preprocessing import label_binarize
import torch.nn as nn
import torch
from tqdm import tqdm

In [ ]:
df_processed = pd.read_csv(Path(RESULTS_DIR, "ADNI_Combined_Multimodal_FIXED.csv"))

CLASSES = df_processed['Group'].unique().tolist()


test_df  = pd.read_csv(Path(RESULTS_DIR, "test_split.csv"))



# Select one sample per class while preserving original indices
selected_df = (
    test_df
    .groupby("Group", group_keys=False)
    .apply(lambda x: x.sample(n=1, random_state=42))
)

# Save the selected samples
selected_df.to_csv(
    Path(RESULTS_DIR, "saliency_selected_samples.csv"),
    index=False
)


test_ds  = ADNIDataset(selected_df,  CLASSES=CLASSES)




batch_size = selected_df.shape[0]


test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

# Sanity check
images, labels = next(iter(test_loader))
print(f"Batch Image Shape: {images.shape}")   # Expect: [B, 1, D, H, W]
print(f"Batch Label Shape: {labels.shape}")   # Expect: [B]
print(f"Sample Labels:     {labels}")

selected_df.head()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DenseNet121(spatial_dims=3, in_channels=1, out_channels=len(CLASSES),dropout_prob=0.1).to(device)


checkpoint = torch.load(Path("./savedmodels", "small3dcnn.pth"), map_location=device)

model.load_state_dict(checkpoint)


target_layer = model.features.denseblock4



In [ ]:
# !pip install grad-cam --quiet

In [ ]:
import numpy as np
import torch
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, ScoreCAM
from pytorch_grad_cam.base_cam import BaseCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget


class GradCAMPlusPlus3D(GradCAMPlusPlus):
    """GradCAM++ patched to handle 5D (N, C, D, H, W) volumetric activations/gradients."""

    def get_cam_weights(self, input_tensor, target_layers, target_category, activations, grads):
        grads_power_2 = grads ** 2
        grads_power_3 = grads_power_2 * grads

        if grads.ndim == 4:
            spatial_axes = (2, 3)
        elif grads.ndim == 5:
            spatial_axes = (2, 3, 4)
        else:
            raise ValueError(f"Unsupported grads shape: {grads.shape}")

        sum_activations = np.sum(activations, axis=spatial_axes)
        eps = 1e-6

        broadcast_shape = sum_activations.shape + (1,) * len(spatial_axes)
        sum_activations = sum_activations.reshape(broadcast_shape)

        aij = grads_power_2 / (2 * grads_power_2 + sum_activations * grads_power_3 + eps)
        aij = np.where(grads != 0, aij, 0)
        weights = np.maximum(grads, 0) * aij
        weights = np.sum(weights, axis=spatial_axes)
        return weights



class ScoreCAM3D(ScoreCAM):
    """
    ScoreCAM patched for 5D (N, C, D, H, W) volumes, with channel-chunked
    upsampling to avoid materializing all activation channels at full
    volume resolution at once (this is what causes CUDA OOM on 3D data).
    """

    def get_cam_weights(self, input_tensor, target_layer, targets, activations, grads):
        with torch.no_grad():
            is_3d = input_tensor.ndim == 5
            spatial_size = input_tensor.shape[-3:] if is_3d else input_tensor.shape[-2:]

            # activations arrives as a numpy array on CPU already (from base_cam) --
            # keep it there and only move small chunks to GPU as needed.
            activation_tensor = torch.from_numpy(activations)  # (N, K, d, h, w) on CPU
            N, K = activation_tensor.shape[0], activation_tensor.shape[1]

            channel_batch_size = getattr(self, "channel_batch_size", 4)
            forward_batch_size = getattr(self, "batch_size", 16)

            all_scores = torch.zeros(N, K)

            for n in range(N):
                img = input_tensor[n : n + 1]      # (1, C, D, H, W)
                target = targets[n]

                for start in range(0, K, channel_batch_size):
                    end = min(start + channel_batch_size, K)

                    acts_chunk = activation_tensor[n : n + 1, start:end].to(self.device)  # (1, k, d, h, w)

                    if is_3d:
                        upsample = torch.nn.Upsample(size=spatial_size, mode="trilinear", align_corners=False)
                    else:
                        upsample = torch.nn.UpsamplingBilinear2d(size=spatial_size)

                    upsampled = upsample(acts_chunk)  # (1, k, D, H, W)

                    flat = upsampled.view(1, upsampled.size(1), -1)
                    maxs = flat.max(dim=-1)[0]
                    mins = flat.min(dim=-1)[0]
                    if is_3d:
                        maxs, mins = maxs[:, :, None, None, None], mins[:, :, None, None, None]
                    else:
                        maxs, mins = maxs[:, :, None, None], mins[:, :, None, None]
                    upsampled = (upsampled - mins) / (maxs - mins + 1e-8)

                    if is_3d:
                        masked = img[:, None, :, :, :, :] * upsampled[:, :, None, :, :, :]  # (1,k,C,D,H,W)
                    else:
                        masked = img[:, None, :, :] * upsampled[:, :, None, :, :]
                    masked = masked[0]  # (k, C, D, H, W)

                    for i in range(0, masked.size(0), forward_batch_size):
                        sub = masked[i : i + forward_batch_size]
                        outputs = self.model(sub)
                        for j, o in enumerate(outputs):
                            all_scores[n, start + i + j] = target(o).cpu().item()

                    del acts_chunk, upsampled, masked
                    torch.cuda.empty_cache()

            weights = torch.nn.Softmax(dim=-1)(all_scores).numpy()
            return weights

In [ ]:
target_layers = [model.features.denseblock4]  # adjust attribute path to your model

gradcampp = GradCAMPlusPlus3D(model=model, target_layers=target_layers)
scorecam = ScoreCAM3D(model=model, target_layers=target_layers)
scorecam = ScoreCAM3D(model=model, target_layers=target_layers)
scorecam.channel_batch_size = 4   # how many activation channels get upsampled at once — lower this first if still OOM
scorecam.batch_size = 4           # how many masked volumes get forwarded through the model at once
# scorecam.batch_size = 8  # tune down if you hit memory limits; each channel = 1 forward pass

# ---------------- one example per class ----------------
def find_one_example_per_class(model, loader, device, num_classes):
    found = {}
    model.eval()
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            preds_batch = model(imgs).argmax(dim=1)
            for i in range(imgs.shape[0]):
                c = int(lbls[i].item())
                if c not in found:
                    found[c] = {
                        "image": imgs[i : i + 1].detach().clone(),
                        "label": c,
                        "pred": int(preds_batch[i].item()),
                    }
            if len(found) == num_classes:
                break
    return found

num_classes = len(CLASSES)
examples = find_one_example_per_class(model, test_loader, device, num_classes)

# ---------------- compute CAMs (batched targets work natively) ----------------
for c, info in examples.items():
    targets = [ClassifierOutputTarget(info["pred"])]
    info["gradcampp"] = gradcampp(input_tensor=info["image"], targets=targets)[0]  # (D, H, W)
    info["scorecam"] = scorecam(input_tensor=info["image"], targets=targets)[0]    # (D, H, W)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# ---------------- Config ----------------
n_slices = 6      # bump to 6, 8, etc. whenever
min_gap = 15
num_classes = len(CLASSES)

# ---------------- Step 1: one example per class ----------------
def find_one_example_per_class(model, loader, device, num_classes):
    found = {}
    model.eval()
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            preds_batch = model(imgs).argmax(dim=1)
            for i in range(imgs.shape[0]):
                c = int(lbls[i].item())
                if c not in found:
                    found[c] = {
                        "image": imgs[i : i + 1].detach().clone(),
                        "label": c,
                        "pred": int(preds_batch[i].item()),
                    }
            if len(found) == num_classes:
                break
    return found

examples = find_one_example_per_class(model, test_loader, device, num_classes)

# ---------------- Step 2: compute both CAMs per example ----------------
for c, info in examples.items():
    targets = [ClassifierOutputTarget(info["pred"])]
    info["gradcampp"] = gradcampp(input_tensor=info["image"], targets=targets)[0]  # (D, H, W)
    info["scorecam"] = scorecam(input_tensor=info["image"], targets=targets)[0]    # (D, H, W)

# ---------------- Step 3: shared slice-selection helper ----------------
def select_representative_slices(examples, cam_key, n_slices, min_gap):
    depth = next(iter(examples.values()))[cam_key].shape[0]

    slice_scores = np.zeros(depth)
    for info in examples.values():
        slice_scores += info[cam_key].clip(min=0).sum(axis=(1, 2))
    slice_scores /= len(examples)

    order = np.argsort(slice_scores)[::-1]
    selected = []
    for idx in order:
        if all(abs(idx - s) >= min_gap for s in selected):
            selected.append(int(idx))
        if len(selected) == n_slices:
            break

    if len(selected) < n_slices:
        remaining = [i for i in order if i not in selected]
        selected.extend(remaining[: n_slices - len(selected)])

    return sorted(selected)

# ---------------- Step 4: shared plotting helper ----------------
def plot_cam_grid(examples, cam_key, title, save_path, n_slices, min_gap):
    slice_indices = select_representative_slices(examples, cam_key, n_slices, min_gap)
    print(f"[{cam_key}] Selected slices:", slice_indices)

    fig, axes = plt.subplots(
        num_classes, n_slices,
        figsize=(4 * n_slices, 4 * num_classes),
        squeeze=False,
    )

    last_im = None
    for row, c in enumerate(sorted(examples.keys())):
        info = examples[c]
        img_vol = info["image"][0, 0].cpu().numpy()   # (D, H, W)
        cam_vol = info[cam_key]                         # (D, H, W)

        correct = info["label"] == info["pred"]

        for col, sl in enumerate(slice_indices):
            ax = axes[row, col]

            base = img_vol[sl]
            heat = cam_vol[sl]
            heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)

            ax.imshow(base, cmap="gray")
            last_im = ax.imshow(heat, cmap="jet", alpha=0.45, vmin=0, vmax=1)

            if row == 0:
                ax.set_title(f"Slice {sl}", fontsize=14, fontweight="bold")

            if col == 0:
                gt_name = CLASSES[info["label"]]
                pred_name = CLASSES[info["pred"]]
                status = "✓" if correct else "✗"
                ax.set_ylabel(
                    f"GT: {gt_name}\nPred: {pred_name} {status}",
                    fontsize=14, rotation=0, labelpad=80, va="center", ha="right",
                )

            ax.set_xticks([])
            ax.set_yticks([])

            if not correct:
                for spine in ax.spines.values():
                    spine.set_edgecolor("red")
                    spine.set_linewidth(2.5)

    # fig.suptitle(title, fontsize=16, fontweight="bold", y=1.02)

    cbar_ax = fig.add_axes([1.01, 0.15, 0.02, 0.7])
    fig.colorbar(last_im, cax=cbar_ax, label="Activation (normalized)")

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

# ---------------- Step 5: render both figures separately ----------------
plot_cam_grid(
    examples, "gradcampp",
    title="Grad-CAM++ Overlays by Class Across Representative Slices",
    save_path="./figures/gradcampp_grid.pdf",
    n_slices=n_slices, min_gap=min_gap,
)

plot_cam_grid(
    examples, "scorecam",
    title="Score-CAM Overlays by Class Across Representative Slices",
    save_path="./figures/scorecam_grid.pdf",
    n_slices=n_slices, min_gap=min_gap,
)

# Multimodal Model Explainability

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, classification_report, roc_auc_score
from tqdm import tqdm
from data import ADNIMultiDataset
from model import MultimodalADNI
from util import preprocess_adni
from config import DATA_DIR, RESULTS_DIR, FIGURS_DIR, METADATA_DIR


In [ ]:
  DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


df_processed = pd.read_csv(Path(RESULTS_DIR, "ADNI_Combined_Multimodal.csv"))
fusion_methods = "concat"  # "cross_attn" concat # Example fusion methods to experiment with

CLASSES = df_processed['Group'].unique().tolist()


train_df = pd.read_csv(Path(RESULTS_DIR, "train_split.csv"))
val_df   = pd.read_csv(Path(RESULTS_DIR, "val_split.csv"))
test_df  = pd.read_csv(Path(RESULTS_DIR, "saliency_selected_samples.csv"))

data = preprocess_adni(train_df, val_df, test_df, keep_paths=True)
classes = data["label_encoder"].classes_

batch_size = 3

# train_dataset = ADNIMultiDataset(train_df, data["X_train"], classes,
#                             data["train_paths"])
# val_dataset   = ADNIMultiDataset(val_df,   data["X_val"],   classes,
#                             data["val_paths"])
test_dataset  = ADNIMultiDataset(test_df,  data["X_test"],  classes,
                            data["test_paths"])


test_loader  = DataLoader(test_dataset, batch_size=batch_size, pin_memory=True, num_workers=0)



n_features = data["X_train"].shape[1]
tab_embed_dim = data["X_test"].shape[-1]
img_embed_dim = 128
fusion_dropout = 0.2

model = MultimodalADNI(
    n_tabular_features=n_features,
    fusion=f"{fusion_methods}",
    img_embed_dim      = img_embed_dim,
    tab_embed_dim      = tab_embed_dim,
    fusion_dropout     = fusion_dropout,
).to(DEVICE)



CHECKPOINT_PATH = Path(RESULTS_DIR, f"checkpoint_{fusion_methods}.pt")


checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

model.load_state_dict(checkpoint['model_state_dict'])

# model.load_state_dict(best_state)


In [ ]:
model.eval()

with torch.no_grad():
    for batch in test_loader:
        img, tab, label = batch

        img = img.to(DEVICE)
        tab = tab.to(DEVICE)

        logits = model(img, tab)
        preds = torch.argmax(logits, dim=1)

        for p, l in zip(preds.cpu(), label):
            print(f"true: {classes[l]} | pred: {classes[p]}")

In [ ]:
target_layers = [model.imaging.backbone.features.denseblock4]  # confirm this path — see below

class MultimodalCAMWrapper(torch.nn.Module):
    """
    Exposes a single-tensor forward(img) for pytorch_grad_cam, while
    holding the tabular vector fixed (set per-example via set_tab).
    """
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.tab_vector = None

    def set_tab(self, tab_vector):
        self.tab_vector = tab_vector

    def forward(self, img):
        b = img.shape[0]
        tab = self.tab_vector.expand(b, -1).to(img.device)
        return self.model(img, tab)


wrapped_model = MultimodalCAMWrapper(model).to(DEVICE)
wrapped_model.eval()

gradcampp = GradCAMPlusPlus3D(model=wrapped_model, target_layers=target_layers)
scorecam  = ScoreCAM3D(model=wrapped_model, target_layers=target_layers)
scorecam.channel_batch_size = 4
scorecam.batch_size = 4
wrapped_model.eval()

gradcampp = GradCAMPlusPlus3D(model=wrapped_model, target_layers=target_layers)
scorecam  = ScoreCAM3D(model=wrapped_model, target_layers=target_layers)
scorecam.channel_batch_size = 4
scorecam.batch_size = 4

# ---------------- one example per class (multimodal) ----------------
def find_one_example_per_class_multimodal(model, loader, device, num_classes):
    found = {}
    model.eval()
    with torch.no_grad():
        for imgs, tabs, lbls in loader:
            imgs, tabs, lbls = imgs.to(device), tabs.to(device), lbls.to(device)
            preds_batch = model(imgs, tabs).argmax(dim=1)
            for i in range(imgs.shape[0]):
                c = int(lbls[i].item())
                if c not in found:
                    found[c] = {
                        "image": imgs[i : i + 1].detach().clone(),
                        "tab":   tabs[i : i + 1].detach().clone(),
                        "label": c,
                        "pred":  int(preds_batch[i].item()),
                    }
            if len(found) == num_classes:
                break
    return found

num_classes = len(CLASSES)
examples = find_one_example_per_class_multimodal(model, test_loader, DEVICE, num_classes)

# ---------------- compute CAMs, swapping the fixed tab vector per example ----------------
for c, info in examples.items():
    wrapped_model.set_tab(info["tab"])
    targets = [ClassifierOutputTarget(info["pred"])]
    info["gradcampp"] = gradcampp(input_tensor=info["image"], targets=targets)[0]  # (D, H, W)
    info["scorecam"]  = scorecam(input_tensor=info["image"], targets=targets)[0]   # (D, H, W)

In [ ]:
def plot_cam_grid(examples, cam_key, title, save_path, n_slices, min_gap):
    slice_indices = select_representative_slices(examples, cam_key, n_slices, min_gap)
    print(f"[{cam_key}] Selected slices:", slice_indices)

    n_rows = len(examples)  # was: num_classes (global) — now matches what was actually found

    fig, axes = plt.subplots(
        n_rows, n_slices,
        figsize=(4 * n_slices, 4 * n_rows),
        squeeze=False,
    )

    last_im = None
    for row, c in enumerate(sorted(examples.keys())):
        info = examples[c]
        img_vol = info["image"][0, 0].cpu().numpy()
        cam_vol = info[cam_key]

        correct = info["label"] == info["pred"]

        for col, sl in enumerate(slice_indices):
            ax = axes[row, col]
            base = img_vol[sl]
            heat = cam_vol[sl]
            heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)

            ax.imshow(base, cmap="gray")
            last_im = ax.imshow(heat, cmap="jet", alpha=0.45, vmin=0, vmax=1)
            # last_im = ax.imshow(heat, cmap="inferno", alpha=0.45, vmin=0, vmax=1)

            if row == 0:
                ax.set_title(f"Slice {sl}", fontsize=14, fontweight="bold")

            if col == 0:
                gt_name = CLASSES[info["label"]]
                pred_name = CLASSES[info["pred"]]
                status = "✓" if correct else "✗"
                ax.set_ylabel(
                    f"GT: {gt_name}\nPred: {pred_name} {status}",
                    fontsize=14, rotation=0, labelpad=80, va="center", ha="right",
                )

            ax.set_xticks([])
            ax.set_yticks([])

            if not correct:
                for spine in ax.spines.values():
                    spine.set_edgecolor("red")
                    spine.set_linewidth(2.5)

    cbar_ax = fig.add_axes([1.01, 0.15, 0.02, 0.7])
    fig.colorbar(last_im, cax=cbar_ax, label="Activation (normalized)")

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

def select_representative_slices(examples, cam_key, n_slices, min_gap):
    depth = next(iter(examples.values()))[cam_key].shape[0]

    slice_scores = np.zeros(depth)
    for info in examples.values():
        slice_scores += info[cam_key].clip(min=0).sum(axis=(1, 2))
    slice_scores /= len(examples)

    order = np.argsort(slice_scores)[::-1]
    selected = []
    for idx in order:
        if all(abs(idx - s) >= min_gap for s in selected):
            selected.append(int(idx))
        if len(selected) == n_slices:
            break

    if len(selected) < n_slices:
        remaining = [i for i in order if i not in selected]
        selected.extend(remaining[: n_slices - len(selected)])

    return sorted(selected)

In [ ]:
n_slices = 6      # bump to 6, 8, etc. whenever
min_gap = 15
num_classes = len(classes)


plot_cam_grid(
    examples, "gradcampp",
    title="Grad-CAM++ Overlays by Class Across Representative Slices",
    save_path="./figures/gradcampp_grid_multimodal.pdf",
    n_slices=n_slices, min_gap=min_gap,
)

plot_cam_grid(
    examples, "scorecam",
    title="Score-CAM Overlays by Class Across Representative Slices",
    save_path="./figures/scorecam_grid_multimodal.pdf",
    n_slices=n_slices, min_gap=min_gap,
)

and the tabular explainability???
The idea is to connect the imaging and tabular explainability to enhance model prediction understanding